# 05 — Affect rises without converging: a dissociation

Emotion rise (Time × Condition), tone→emotion, connectedness→emotion, and emotion non-convergence (permutation + pseudo-group, 10,000 resamples).

**Expected:** time main effect β=6.23, 95%CI[3.36,9.10], p<.001; time×condition β=5.85, 95%CI[0.11,11.59], p=.046 (collab +9.16 vs indiv +3.31). tone→emotion r=.65, permutation p=.014. connectedness→emotion between β=+18.11, p=.008; within β=+2.61, p=.56. Non-convergence: observed +0.74; permutation null +1.03 (SD 1.44), p=.40; pseudo-group −1.05 (SD 1.21), p=.93. Affect–belief coupling: individual related r=.30 p=.010, unrelated r=.09 p=.43, Δr=.20 bootstrap p=.046; collaborative related r=.46 p<.001, unrelated r=.32 p=.004, Δr=.14 p=.25, mixed b=0.63 p<.001.

In [1]:
import pandas as pd, numpy as np, warnings
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
DATA = '../data'
def load_belief(): return pd.read_excel(f'{DATA}/belief_items.xlsx')
def load_subject(): return pd.read_excel(f'{DATA}/subject_measures.xlsx')
def load_group(): return pd.read_csv(f'{DATA}/group_conversation.csv')
def rep(m,t):
    ci=m.conf_int().loc[t]
    return f'b={m.params[t]:+.3f} 95%CI[{ci[0]:+.3f}, {ci[1]:+.3f}] p={m.pvalues[t]:.4f}'
def corr_ci(x,y):
    r,p=stats.pearsonr(x,y); n=len(x); z=np.arctanh(r); se=1/np.sqrt(n-3)
    return r,np.tanh(z-1.96*se),np.tanh(z+1.96*se),p,n
from itertools import combinations
SEED=20260725; NP=10000


In [2]:
s=load_subject(); E=s[['participant','condition','group','emotion_pre','emotion_post']].dropna()
L=E.melt(['participant','condition','group'],['emotion_pre','emotion_post'],'t','em')
L['time_n']=(L.t=='emotion_post').astype(int)
m=smf.mixedlm('em ~ time_n*C(condition, Sum)', L, groups=L.participant).fit()
print('time main effect:', rep(m,'time_n'))
print(f'  all participants: pre M={E.emotion_pre.mean():.1f}, post M={E.emotion_post.mean():.1f} (n={len(E)})')
# time x condition as the full contrast (dummy condition, individual ref)
L['collab']=(L.condition=='collaborative').astype(int)
m2=smf.mixedlm('em ~ time_n*collab', L, groups=L.participant).fit()
print('time x condition:', rep(m2,'time_n:collab'))
for cond in ['collaborative','individual']:
    ch=(E[E.condition==cond].emotion_post-E[E.condition==cond].emotion_pre).mean(); print(f'  {cond} increase = {ch:+.2f}')


time main effect: b=+6.232 95%CI[+3.342, +9.122] p=0.0000
  all participants: pre M=49.0, post M=55.3 (n=151)
time x condition: b=+5.851 95%CI[+0.072, +11.631] p=0.0472
  collaborative increase = +9.16
  individual increase = +3.31


In [3]:
# tone -> emotion
g=load_group(); gem=s[s.condition=='collaborative'].assign(ec=lambda x:x.emotion_post-x.emotion_pre).groupby('group')['ec'].mean()
G=g.set_index('group')[['tone']].join(gem.rename('emo')).dropna()
rng=np.random.default_rng(SEED); tv=G.tone.values; y=G.emo.values
obs=np.corrcoef(tv,y)[0,1]; nul=np.array([np.corrcoef(rng.permutation(tv),y)[0,1] for _ in range(NP)])
r,lo,hi,_,n=corr_ci(G.tone,G.emo)
print(f'tone->emotion r={r:+.2f} 95%CI[{lo:+.2f},{hi:+.2f}] permutation p={(np.sum(nul>=obs)+1)/(NP+1):.3f} (null {nul.mean():+.2f}, SD {nul.std():.2f}) n={n}')
# connectedness -> emotion (between/within)
c=s[s.condition=='collaborative'][['group','conn_mean','emotion_change']].dropna()
c['gm']=c.groupby('group').conn_mean.transform('mean'); c['wi']=c.conn_mean-c.gm
me=smf.mixedlm('emotion_change ~ gm + wi', c, groups=c.group).fit()
print('conn->emotion between:', rep(me,'gm')); print('conn->emotion within:', rep(me,'wi'))


tone->emotion r=+0.65 95%CI[+0.25,+0.86] permutation p=0.012 (null +0.00, SD 0.25) n=17
conn->emotion between: b=+18.111 95%CI[+4.664, +31.558] p=0.0083
conn->emotion within: b=+2.614 95%CI[-6.070, +11.297] p=0.5553


In [4]:
# emotion non-convergence (shuffle post values among collaborative; pseudo-groups from individuals)
def gconv(idx,Dpre,Dpost):
    ds=[]
    for m2 in idx:
        if len(m2)<2: continue
        pr=[(i,j) for i,j in combinations(m2,2)]; ds.append(np.mean([Dpost[i,j] for i,j in pr])-np.mean([Dpre[i,j] for i,j in pr]))
    return np.mean(ds)
exp=E[E.condition=='collaborative'].reset_index(drop=True); ctrl=E[E.condition=='individual'].reset_index(drop=True)
pre=exp.emotion_pre.values; post=exp.emotion_post.values; Dpre=np.abs(pre[:,None]-pre[None,:])
gmap={}
for i,gg in enumerate(exp.group.values): gmap.setdefault(gg,[]).append(i)
real=[v for v in gmap.values() if len(v)>=2]; sizes=[len(v) for v in real]
obs=gconv(real,Dpre,np.abs(post[:,None]-post[None,:]))
rng=np.random.default_rng(SEED); perm=np.empty(NP)
for k in range(NP):
    sp=rng.permutation(post); perm[k]=gconv(real,Dpre,np.abs(sp[:,None]-sp[None,:]))
cpre=ctrl.emotion_pre.values; cpost=ctrl.emotion_post.values; cDpre=np.abs(cpre[:,None]-cpre[None,:]); cDpost=np.abs(cpost[:,None]-cpost[None,:])
cn=len(cpre); n4=cn//4; cidx=np.arange(cn); ps=np.empty(NP)
for k in range(NP):
    pp=rng.permutation(cidx); ps[k]=gconv([list(pp[i*4:(i+1)*4]) for i in range(n4)],cDpre,cDpost)
print(f'emotion non-convergence: observed={obs:+.2f} | permutation null {perm.mean():+.2f} (SD {perm.std():.2f}) p={(np.sum(perm<=obs)+1)/(NP+1):.3f}'
      f' | pseudo-group {ps.mean():+.2f} (SD {ps.std():.2f}) p={(np.sum(ps<=obs)+1)/(NP+1):.3f}')


emotion non-convergence: observed=+0.74 | permutation null +1.02 (SD 1.44) p=0.397 | pseudo-group -1.07 (SD 1.22) p=0.930


In [5]:
# affective change ~ own belief change, by condition (Results, final paragraph of the affect section)
s=load_subject()
s=s[s.in_belief_emotion==1] if 'in_belief_emotion' in s else s
rng=np.random.default_rng(20260828); NB=10000
for cond in ['individual','collaborative']:
    d=s[s.condition==cond].dropna(subset=['emotion_change','bel_related_change','bel_unrelated_change'])
    print(f'{cond.upper()} (n={len(d)}' + (f', {d.group.nunique()} groups)' if cond=='collaborative' else ')'))
    rr=corr_ci(d.emotion_change,d.bel_related_change); ru=corr_ci(d.emotion_change,d.bel_unrelated_change)
    print(f'  Demotion ~ Dbelief RELATED    r={rr[0]:+.3f} 95%CI[{rr[1]:+.3f}, {rr[2]:+.3f}] p={rr[3]:.4f}')
    print(f'  Demotion ~ Dbelief UNRELATED  r={ru[0]:+.3f} 95%CI[{ru[1]:+.3f}, {ru[2]:+.3f}] p={ru[3]:.4f}')
    m=smf.ols('emotion_change ~ bel_related_change + bel_unrelated_change', d).fit()
    for t in ['bel_related_change','bel_unrelated_change']: print(f'  joint {t:22s}', rep(m,t))
    dif=[]
    if cond=='collaborative':
        gl=[grp for _,grp in d.groupby('group')]
        for _ in range(NB):
            bs=pd.concat([gl[i] for i in rng.integers(0,len(gl),len(gl))])
            dif.append(np.corrcoef(bs.emotion_change,bs.bel_related_change)[0,1]-np.corrcoef(bs.emotion_change,bs.bel_unrelated_change)[0,1])
    else:
        v=d[['emotion_change','bel_related_change','bel_unrelated_change']].values
        for _ in range(NB):
            bs=v[rng.integers(0,len(v),len(v))]
            dif.append(np.corrcoef(bs[:,0],bs[:,1])[0,1]-np.corrcoef(bs[:,0],bs[:,2])[0,1])
    dif=np.array(dif); obs=rr[0]-ru[0]
    print(f'  content-specificity Dr={obs:+.3f} bootstrap 95%CI[{np.percentile(dif,2.5):+.3f}, {np.percentile(dif,97.5):+.3f}] p={2*min((dif<=0).mean(),(dif>=0).mean()):.3f}')
    if cond=='collaborative':
        mm=smf.mixedlm('emotion_change ~ bel_related_change', d, groups=d.group).fit()
        print('  mixed model (RI group)  ', rep(mm,'bel_related_change'))
    print()

INDIVIDUAL (n=75)
  Demotion ~ Dbelief RELATED    r=+0.296 95%CI[+0.074, +0.490] p=0.0100
  Demotion ~ Dbelief UNRELATED  r=+0.093 95%CI[-0.136, +0.314] p=0.4252
  joint bel_related_change     b=+0.337 95%CI[+0.068, +0.605] p=0.0147
  joint bel_unrelated_change   b=-0.034 95%CI[-0.391, +0.323] p=0.8500
  content-specificity Dr=+0.202 bootstrap 95%CI[+0.003, +0.408] p=0.046

COLLABORATIVE (n=76, 19 groups)
  Demotion ~ Dbelief RELATED    r=+0.463 95%CI[+0.266, +0.624] p=0.0000
  Demotion ~ Dbelief UNRELATED  r=+0.323 95%CI[+0.105, +0.511] p=0.0044
  joint bel_related_change     b=+0.543 95%CI[+0.238, +0.848] p=0.0007
  joint bel_unrelated_change   b=+0.267 95%CI[-0.102, +0.636] p=0.1540
  content-specificity Dr=+0.140 bootstrap 95%CI[-0.096, +0.434] p=0.255
  mixed model (RI group)   b=+0.630 95%CI[+0.350, +0.910] p=0.0000

